# Mejora 7 - Corrección de la cláusula final, NB-LR y modelo en dos etapas con NB-LR

**Proyecto - Parte 1: Machine Learning clásico**

**Estudiantes:**

[Nombre integrante 1] – [código]

[Nombre integrante 2] – [código]

**Curso:** ISIS2611 - Aprendizaje de Máquina

---

## Contexto

Los modelos de los notebooks anteriores tienen una exactitud real de entre 0,897 y 0,901 (medida con 15 particiones) y puntajes públicos en Kaggle de entre 0,899 y 0,907. Este notebook hace tres cosas. Primero corrige un error de la representación por bloques que venía desde el notebook principal (sección 4). Después prueba una técnica clásica de clasificación de sentimiento que todavía no se había usado: **NB-LR**, una regresión logística sobre la bolsa de palabras ponderada con las razones de Naive Bayes (Wang y Manning, 2012, *Baselines and Bigrams*). Por último combina NB-LR con el modelo en dos etapas de la Mejora 1.

Como la métrica de la competencia es la exactitud, en este notebook los modelos se eligen y se comparan por exactitud (en los anteriores se usaba el F1-macro, que en estos datos da resultados muy parecidos).

## Requisitos de la guía

| Requisito | Cómo se cumple |
|:---|:---|
| No usar aprendizaje profundo, transformers ni embeddings preentrenados. | No se usa ninguno. Todo se basa en bolsas de palabras, TF-IDF y n-gramas. |
| Usar únicamente modelos de scikit-learn. | Los modelos son `LogisticRegression` y `MultinomialNB`. `NBRegresionLogistica` y `ModeloDosEtapasNBLR` son clases propias que solo combinan esos modelos con la interfaz de scikit-learn. |
| Resultados replicables. | Semillas fijas, versiones en `requirements.txt` y todo el flujo en este notebook (sección 14). |

NB-LR combina dos modelos vistos en clase (Naive Bayes y regresión logística) y el modelo en dos etapas encadena modelos. Ambas ideas van un paso más allá de lo visto en el curso; si el equipo considera que no encajan, el modelo del notebook principal sigue siendo la opción más simple.

## 1. Importación de librerías

In [1]:
import os
import re
import unicodedata
import warnings
from collections import Counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from nltk.stem import SnowballStemmer
from scipy import sparse

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, ConfusionMatrixDisplay,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)

RANDOM_STATE = 42
ORDEN_CLASES = ['negativo', 'neutral', 'positivo']

# Rutas del proyecto (este notebook está en la carpeta notebooks/)
RUTA_DATOS = os.path.join('..', 'data')
RUTA_MODELOS = os.path.join('..', 'models')
RUTA_ENVIOS = os.path.join('..', 'submissions')
NOMBRE_MODELO = 'dos_etapas_nblr'

os.makedirs(RUTA_MODELOS, exist_ok=True)
os.makedirs(RUTA_ENVIOS, exist_ok=True)
NOMBRE_NBLR = 'nb_regresion_logistica'
NOMBRE_CORREGIDO = 'regresion_logistica_corregida'

## 2. Datos y partición

Se usan los mismos datos, la misma partición 80/20 estratificada (semilla 42) y las mismas 5 particiones de validación cruzada de los notebooks anteriores, de modo que los resultados son comparables.

In [2]:
train = pd.read_csv(os.path.join(RUTA_DATOS, 'train.csv'))
evaluacion = pd.read_csv(os.path.join(RUTA_DATOS, 'eval.csv'))
muestra_envio = pd.read_csv(os.path.join(RUTA_DATOS, 'sample_submission.csv'))

print('train:', train.shape, '| eval:', evaluacion.shape, '| sample_submission:', muestra_envio.shape)
print('Distribución de clases en train:')
display(train['label'].value_counts(normalize=True).reindex(ORDEN_CLASES).round(3).to_frame('proporción'))

train: (12000, 3) | eval: (3000, 2) | sample_submission: (3000, 2)
Distribución de clases en train:


,proporción
label,
negativo,0.351
neutral,0.298
positivo,0.351


In [3]:
X = train[['text']]
y = train['label']

# Misma partición y mismas particiones de validación cruzada que en el notebook principal
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)

Entrenamiento: (9600, 1) | Prueba: (2400, 1)


## 3. Preprocesamiento y modelo actual de referencia

In [4]:
stemmer = SnowballStemmer('spanish')
cache_raices = {}


def raiz(palabra):
    # Se guarda la raíz de cada palabra para no repetir el stemming miles de veces
    if palabra not in cache_raices:
        cache_raices[palabra] = stemmer.stem(palabra)
    return cache_raices[palabra]


def quitar_tildes(texto):
    texto = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in texto if not unicodedata.combining(c))


def limpiar_texto(texto, stop=None, stemming=True):
    texto = quitar_tildes(texto.lower())
    texto = re.sub(r'[^\w\s]', ' ', texto)
    texto = re.sub(r'\d+', ' ', texto)
    tokens = texto.split()
    if stop:
        tokens = [t for t in tokens if t not in stop]
    if stemming:
        tokens = [raiz(t) for t in tokens]
    return ' '.join(tokens)


def separar_oraciones(texto):
    return [o.strip() for o in re.split(r'(?<=[.!?])\s+', texto) if o.strip()]


# Conectores que introducen la opinión que se impone sobre la anterior
CONECTORES = r'\b(?:pero|aunque|sin embargo|aun asi|con todo|a fin de cuentas|al final)\b'


def extraer_completo(textos, stemming=True):
    return [limpiar_texto(t, stemming=stemming) for t in textos]


def extraer_ultima(textos, stemming=True):
    return [limpiar_texto(separar_oraciones(t)[-1], stemming=stemming) for t in textos]


def extraer_clausula_final(textos, stemming=True):
    salida = []
    for t in textos:
        ultima = quitar_tildes(separar_oraciones(t)[-1].lower())
        salida.append(limpiar_texto(re.split(CONECTORES, ultima)[-1], stemming=stemming))
    return salida


def bloque_tfidf(nombre, extractor, rango=(1, 2)):
    return (nombre,
            Pipeline([('extraer', FunctionTransformer(extractor)),
                      ('tfidf', TfidfVectorizer(ngram_range=rango))]),
            'text')


def construir_representacion(bloques=('completo', 'ultima', 'clausula'), rango=(1, 2)):
    extractores = {'completo': extraer_completo,
                   'ultima': extraer_ultima,
                   'clausula': extraer_clausula_final}
    return ColumnTransformer([bloque_tfidf(b, extractores[b], rango) for b in bloques])

In [5]:
def metricas(y_real, y_pred):
    return {'Exactitud': accuracy_score(y_real, y_pred),
            'Precisión (macro)': precision_score(y_real, y_pred, average='macro'),
            'Recall (macro)': recall_score(y_real, y_pred, average='macro'),
            'F1 (macro)': f1_score(y_real, y_pred, average='macro')}

In [6]:
# Modelo actual (el del notebook principal): regresión logística L1 sobre la representación por bloques
modelo_actual = Pipeline([
    ('rep', construir_representacion()),
    ('modelo', LogisticRegression(C=10, penalty='l1', solver='liblinear', max_iter=3000, random_state=RANDOM_STATE)),
])
modelo_actual.fit(X_train, y_train)
pred_actual = pd.Series(modelo_actual.predict(X_test), index=y_test.index)

cv_actual = cross_val_score(modelo_actual, X_train, y_train, cv=kfold, scoring='accuracy', n_jobs=-1)
print(f'Modelo actual - Exactitud en validación cruzada (train): {cv_actual.mean():.4f} +/- {cv_actual.std():.4f}')
pd.DataFrame({'Modelo actual (test)': metricas(y_test, pred_actual)}).T.round(4)

Modelo actual - Exactitud en validación cruzada (train): 0.8948 +/- 0.0074


,Exactitud,Precisión (macro),Recall (macro),F1 (macro)
Modelo actual (test),0.8946,0.8987,0.8998,0.8993


## 4. Corrección de la cláusula final

Al revisar los errores en reseñas con conectores de contraste apareció un problema de la representación del notebook principal. La lista de conectores incluye "al final" y "a fin de cuentas", pero en estos datos esas expresiones aparecen muchas veces como **muletilla al final de la frase** ("la batería se mantuvo decente al final de cuentas"). Al cortar ahí, el bloque de cláusula final se quedaba solo con "de cuentas" y perdía la opinión.

La corrección tiene dos partes:

1. No cortar en "al final de cuentas" ni en "a fin de cuentas". "Al final" sigue siendo conector cuando no va seguido de "de cuentas".
2. Si después del corte el último fragmento tiene menos de 3 palabras, se toma el fragmento anterior.

In [7]:
# "al final de cuentas" y "a fin de cuentas" suelen ser muletillas al final de la frase, no conectores.
# Se deja de cortar ahí y, si el corte deja un fragmento de menos de 3 palabras, se toma el anterior.
CONECTORES_CORREGIDOS = r'\b(?:pero|aunque|sin embargo|aun asi|con todo|al final(?! de cuentas))\b'


def ultimo_fragmento(oracion, min_palabras=3):
    partes = [p for p in re.split(CONECTORES_CORREGIDOS, quitar_tildes(oracion.lower())) if len(p.split()) >= min_palabras]
    return partes[-1] if partes else oracion


def extraer_clausula_corregida(textos, stemming=True):
    return [limpiar_texto(ultimo_fragmento(separar_oraciones(t)[-1]), stemming=stemming) for t in textos]


def construir_representacion_corregida(rango=(1, 2)):
    return ColumnTransformer([bloque_tfidf('completo', extraer_completo, rango),
                              bloque_tfidf('ultima', extraer_ultima, rango),
                              bloque_tfidf('clausula', extraer_clausula_corregida, rango)])


cambian = [t for t in X_train['text'] if extraer_clausula_final([t]) != extraer_clausula_corregida([t])]
print(f'Reseñas de train cuya cláusula final cambia: {len(cambian)} de {len(X_train)}')
ejemplo = next(t for t in cambian if 'de cuentas' in quitar_tildes(t.lower()))
print('\nÚLTIMA ORACIÓN:', separar_oraciones(ejemplo)[-1])
print('Cláusula antes:   ', extraer_clausula_final([ejemplo])[0])
print('Cláusula después: ', extraer_clausula_corregida([ejemplo])[0])

Reseñas de train cuya cláusula final cambia: 334 de 9600

ÚLTIMA ORACIÓN: Al final el cable me resulto ser de buena calidad al final de cuentas.
Cláusula antes:    de cuent
Cláusula después:  el cabl me result ser de buen calid al final de cuent


In [8]:
modelo_corregido = Pipeline([
    ('rep', construir_representacion_corregida()),
    ('modelo', LogisticRegression(C=10, penalty='l1', solver='liblinear', max_iter=3000, random_state=RANDOM_STATE)),
])
cv_corregido = cross_val_score(modelo_corregido, X_train, y_train, cv=kfold, scoring='accuracy', n_jobs=-1)
modelo_corregido.fit(X_train, y_train)
pred_corregido = pd.Series(modelo_corregido.predict(X_test), index=y_test.index)
print(f'Exactitud en validación cruzada -> modelo actual: {cv_actual.mean():.4f} | corregido: {cv_corregido.mean():.4f}')
pd.DataFrame({'Modelo actual': metricas(y_test, pred_actual),
              'Modelo actual con la cláusula corregida': metricas(y_test, pred_corregido)}).T.round(4)

Exactitud en validación cruzada -> modelo actual: 0.8948 | corregido: 0.8966


,Exactitud,Precisión (macro),Recall (macro),F1 (macro)
Modelo actual,0.8946,0.8987,0.8998,0.8993
Modelo actual con la cláusula corregida,0.8954,0.8995,0.9005,0.9000


La corrección cambia la cláusula final de 334 de las 9.600 reseñas de entrenamiento. Con el mismo modelo del notebook principal, la exactitud en validación cruzada pasa de 0,8948 a 0,8966 y en el conjunto de prueba de 0,8946 a 0,8954. En la comparación de 15 particiones (sección 8) la mejora es de 0,21 puntos y se da en las 15 particiones, así que es real aunque sea pequeña. A partir de aquí, todos los modelos de este notebook usan la cláusula corregida.

## 5. NB-LR: regresión logística con atributos ponderados por Naive Bayes

NB-LR parte de la misma representación por bloques del notebook principal (texto completo, última oración y cláusula final, con unigramas y bigramas), pero con **presencia o ausencia** de cada n-grama en lugar de TF-IDF. Luego, para cada clase (una contra el resto):

1. Se calcula, como en Naive Bayes multinomial, qué tan frecuente es cada n-grama en las reseñas de la clase y en las del resto (con suavizado `alpha`).
2. Cada atributo se multiplica por el logaritmo de la razón entre esas dos frecuencias. Los n-gramas muy característicos de una clase quedan con un peso grande (positivo o negativo) y los que aparecen por igual en todas las clases quedan cerca de cero.
3. Sobre esos atributos ponderados se entrena una regresión logística.

La clase predicha es la de mayor probabilidad entre los tres modelos. La idea es que Naive Bayes aporta una buena estimación de cuánto "pesa" cada n-grama, sobre todo con pocos datos, y la regresión logística corrige su supuesto de independencia entre palabras. Se implementa como una clase con la interfaz de scikit-learn, de modo que se puede usar dentro de un `Pipeline` y con `GridSearchCV`.

In [9]:
def construir_bolsas_binarias(rango=(1, 2)):
    # Los mismos tres bloques del notebook principal, pero con presencia/ausencia de cada n-grama en lugar de TF-IDF
    extractores = {'completo': extraer_completo, 'ultima': extraer_ultima, 'clausula': extraer_clausula_corregida}
    return ColumnTransformer([
        (nombre, Pipeline([('extraer', FunctionTransformer(extractor)),
                           ('conteo', CountVectorizer(ngram_range=rango, binary=True))]), 'text')
        for nombre, extractor in extractores.items()
    ])


class NBRegresionLogistica(ClassifierMixin, BaseEstimator):
    """Regresión logística sobre atributos ponderados con la razón de Naive Bayes (NB-LR).

    Para cada clase (una contra el resto) cada atributo se multiplica por
    log(P(atributo | clase) / P(atributo | resto)), calculado con suavizado alpha como en
    MultinomialNB, y sobre esos atributos se entrena una regresión logística.
    """

    def __init__(self, C=0.3, alpha=1.0):
        self.C = C
        self.alpha = alpha

    def fit(self, X, y):
        X = sparse.csr_matrix(X, dtype=float)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.razones_, self.modelos_ = [], []
        for clase in self.classes_:
            es_clase = y == clase
            p = self.alpha + np.asarray(X[es_clase].sum(axis=0)).ravel()
            q = self.alpha + np.asarray(X[~es_clase].sum(axis=0)).ravel()
            razon = np.log((p / p.sum()) / (q / q.sum()))
            modelo = LogisticRegression(C=self.C, max_iter=3000)
            modelo.fit(X.multiply(razon).tocsr(), es_clase.astype(int))
            self.razones_.append(razon)
            self.modelos_.append(modelo)
        return self

    def predict_proba(self, X):
        X = sparse.csr_matrix(X, dtype=float)
        P = np.column_stack([m.predict_proba(X.multiply(r).tocsr())[:, 1]
                             for m, r in zip(self.modelos_, self.razones_)])
        return P / P.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

In [10]:
pipe_nblr = Pipeline([('rep', construir_bolsas_binarias()), ('modelo', NBRegresionLogistica())])

grilla_nblr = {
    'modelo__C': [0.1, 0.2, 0.3, 0.5, 1],
    'modelo__alpha': [0.5, 1.0],
}
busqueda_nblr = GridSearchCV(pipe_nblr, grilla_nblr, cv=kfold, scoring='accuracy', n_jobs=-1)
busqueda_nblr.fit(X_train, y_train)

print('Mejores hiperparámetros:', busqueda_nblr.best_params_)
print(f'Exactitud en validación cruzada: {busqueda_nblr.best_score_:.4f}   (modelo actual: {cv_actual.mean():.4f})')
res_nblr = pd.DataFrame(busqueda_nblr.cv_results_)
res_nblr[['param_modelo__C', 'param_modelo__alpha', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score').head(6).round(4)

Mejores hiperparámetros: {'modelo__C': 0.1, 'modelo__alpha': 0.5}
Exactitud en validación cruzada: 0.8970   (modelo actual: 0.8948)


,param_modelo__C,param_modelo__alpha,mean_test_score,std_test_score,rank_test_score
0,0.1,0.5,0.8970,0.0079,1
2,0.2,0.5,0.8964,0.0089,2
4,0.3,0.5,0.8964,0.0082,2
7,0.5,1.0,0.8957,0.0086,4
5,0.3,1.0,0.8956,0.0070,5
6,0.5,0.5,0.8955,0.0072,6


In [11]:
# ¿Qué n-gramas pesan más? Razón de Naive Bayes de la clase "positivo" frente al resto
mejor_nblr = busqueda_nblr.best_estimator_
rep_nblr = mejor_nblr.named_steps['rep']
nombres = []
for bloque in ['completo', 'ultima', 'clausula']:
    vocabulario = rep_nblr.named_transformers_[bloque].named_steps['conteo'].get_feature_names_out()
    nombres.extend([f'{bloque}: {t}' for t in vocabulario])
nb_lr = mejor_nblr.named_steps['modelo']
i_pos = list(nb_lr.classes_).index('positivo')
razon_pos = pd.Series(nb_lr.razones_[i_pos], index=nombres)
print('Número de atributos:', len(nombres))
display(pd.DataFrame({'más asociados a positivo': razon_pos.sort_values(ascending=False).head(10).index,
                      'más asociados a no positivo': razon_pos.sort_values().head(10).index}))

Número de atributos: 71169


,más asociados a positivo,más asociados a no positivo
0,clausula: impec,ultima: tra garanti
1,clausula: sobresalient,clausula: tra garanti
2,clausula: potent,clausula: terribl
3,clausula: eficient,clausula: pobr
4,clausula: confiabl,ultima: dispon
5,clausula: decent,clausula: dispon
6,clausula: buenisim,clausula: dispon en
7,clausula: veloz,clausula: torp
8,clausula: excelent,clausula: sol lo
9,clausula: optim,clausula: imprecis


La mejor configuración de NB-LR usa `C = 0,1` y `alpha = 0,5`, con una exactitud de 0,8970 en validación cruzada frente a 0,8948 del modelo actual. Las seis mejores configuraciones están entre 0,8955 y 0,8970, así que la elección exacta de los hiperparámetros importa poco.

Los n-gramas con mayor razón hacia `positivo` son adjetivos de valoración en la cláusula final ("impecable", "sobresaliente", "potente", "eficiente", "confiable", "excelente"). Del lado contrario aparecen tanto palabras negativas ("terrible", "pobre", "torpe", "impreciso") como marcas típicas de las reseñas neutrales ("trae garantía", "disponible"), porque el modelo de la clase `positivo` se entrena contra todas las demás reseñas, negativas y neutrales.

## 6. Modelo en dos etapas con NB-LR

El modelo en dos etapas de la Mejora 1 parte cada reseña en trozos, los puntúa con un primer clasificador y decide la etiqueta con una regresión logística sobre los puntajes de los trozos y las probabilidades de los modelos por bloques. Aquí se le agregan, como atributos de la segunda etapa, las **probabilidades de NB-LR**. El resto del modelo es igual al de la Mejora 1 (ver ese notebook para la explicación detallada de cada atributo y de por qué los atributos se calculan fuera de muestra).

Se busca el parámetro de regularización de la segunda etapa (`C_etapa2`); los demás quedan fijos en los valores que funcionaron en las mejoras anteriores (`C_etapa1 = 3`, y para NB-LR `C = 0,3` y `alpha = 1`).

In [12]:
K_ULTIMOS = 4          # cuántos de los últimos trozos se usan como atributos
N_INICIOS = 12         # cuántos comienzos de oración frecuentes se usan como atributos


def separar_trozos(texto):
    """Parte cada oración en trozos usando los conectores de contraste.
    Devuelve una lista de (trozo, tras_conector); tras_conector vale 1 si el trozo viene después de un conector."""
    trozos = []
    for oracion in separar_oraciones(texto):
        partes = re.split(CONECTORES_CORREGIDOS, quitar_tildes(oracion.lower()))
        for i, parte in enumerate(partes):
            if parte.strip():
                trozos.append((parte.strip(), int(i > 0)))
    return trozos


def entrenar_etapa1(textos, etiquetas, C):
    trozos, etiquetas_trozos = [], []
    for texto, etiqueta in zip(textos, etiquetas):
        for trozo, _ in separar_trozos(texto):
            trozos.append(limpiar_texto(trozo))
            etiquetas_trozos.append(etiqueta)    # cada trozo hereda la etiqueta de su reseña
    vectorizador = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    clasificador = LogisticRegression(C=C, max_iter=2000)
    clasificador.fit(vectorizador.fit_transform(trozos), etiquetas_trozos)
    return vectorizador, clasificador


def comienzo_oracion(oracion):
    palabras = re.sub(r'[^\w\s]', '', quitar_tildes(oracion.lower())).split()
    return ' '.join(palabras[:2])


def comienzos_frecuentes(textos, n=N_INICIOS):
    conteo = Counter(comienzo_oracion(o) for t in textos for o in separar_oraciones(t)[1:])
    return [c for c, _ in conteo.most_common(n)]

In [13]:
class ModeloDosEtapasNBLR(ClassifierMixin, BaseEstimator):
    """Modelo en dos etapas de la Mejora 1 al que se le agregan las probabilidades de NB-LR.

    Etapa 1: puntúa cada trozo de la reseña (probabilidad de negativo, neutral y positivo).
    Etapa 2: regresión logística sobre los puntajes de los últimos trozos, las probabilidades de los
    modelos por bloques (regresión logística, Naive Bayes y NB-LR) y el comienzo de las oraciones con opinión.
    """

    def __init__(self, C_etapa1=3, C_etapa2=1, C_nblr=0.3, alpha_nblr=1.0, n_particiones=5, random_state=42):
        self.C_etapa1 = C_etapa1
        self.C_etapa2 = C_etapa2
        self.C_nblr = C_nblr
        self.alpha_nblr = alpha_nblr
        self.n_particiones = n_particiones
        self.random_state = random_state

    def _entrenar_componentes(self, textos, etiquetas):
        datos = pd.DataFrame({'text': textos.values})
        representacion = construir_representacion_corregida().fit(datos)
        matriz = representacion.transform(datos)
        regresion = LogisticRegression(C=10, penalty='l1', solver='liblinear', max_iter=3000,
                                       random_state=RANDOM_STATE).fit(matriz, etiquetas)
        bayes = MultinomialNB(alpha=0.3).fit(matriz, etiquetas)
        nblr = Pipeline([('rep', construir_bolsas_binarias()),
                         ('modelo', NBRegresionLogistica(C=self.C_nblr, alpha=self.alpha_nblr))]).fit(datos, etiquetas)
        vectorizador, clasificador = entrenar_etapa1(textos, etiquetas, self.C_etapa1)
        return {'representacion': representacion, 'regresion': regresion, 'bayes': bayes, 'nblr': nblr,
                'vectorizador': vectorizador, 'clasificador': clasificador}

    def _caracteristicas(self, textos, comp):
        datos = pd.DataFrame({'text': textos.values})
        matriz = comp['representacion'].transform(datos)
        prob_regresion = comp['regresion'].predict_proba(matriz)
        prob_bayes = comp['bayes'].predict_proba(matriz)
        prob_nblr = comp['nblr'].predict_proba(datos)

        trozos_por_texto = [separar_trozos(t) for t in textos]
        oraciones_por_texto = [separar_oraciones(t) for t in textos]
        probs_trozos = comp['clasificador'].predict_proba(comp['vectorizador'].transform(
            [limpiar_texto(trozo) for lista in trozos_por_texto for trozo, _ in lista]))
        probs_oraciones = comp['clasificador'].predict_proba(comp['vectorizador'].transform(
            [limpiar_texto(o) for lista in oraciones_por_texto for o in lista]))

        filas = []
        ini_t = ini_o = 0
        for k in range(len(textos)):
            trozos, oraciones = trozos_por_texto[k], oraciones_por_texto[k]
            P = probs_trozos[ini_t:ini_t + len(trozos)]
            Po = probs_oraciones[ini_o:ini_o + len(oraciones)]
            ini_t += len(trozos)
            ini_o += len(oraciones)

            fila = list(prob_regresion[k])
            for j in range(1, K_ULTIMOS + 1):
                if j <= len(trozos):
                    fila += list(P[-j]) + [trozos[-j][1]]
                else:
                    fila += [0, 0, 0, 0]
            polaridad = P[:, 2] - P[:, 0]
            con_opinion = np.where(P[:, 1] < 0.5)[0]
            fila += [polaridad[con_opinion[-1]] if len(con_opinion) > 0 else 0,
                     polaridad[con_opinion[-2]] if len(con_opinion) > 1 else 0,
                     polaridad[con_opinion[0]] if len(con_opinion) > 0 else 0,
                     len(con_opinion), P[:, 2].max(), P[:, 0].max(), P[:, 1].mean(), len(trozos)]
            fila += list(prob_bayes[k])
            oraciones_con_opinion = np.where(Po[:, 1] < 0.5)[0]
            for pos in [-1, -2]:
                comienzo = ''
                if len(oraciones_con_opinion) >= -pos:
                    comienzo = comienzo_oracion(oraciones[oraciones_con_opinion[pos]])
                fila += [int(comienzo == c) for c in self.comienzos_]
            fila += list(prob_nblr[k])
            filas.append(fila)
        return np.array(filas)

    def fit(self, X, y):
        textos = pd.Series(np.asarray(X['text']))
        etiquetas = pd.Series(np.asarray(y))
        self.classes_ = np.array(sorted(etiquetas.unique()))
        self.comienzos_ = comienzos_frecuentes(textos)

        # Atributos de la etapa 2 calculados fuera de muestra (validación cruzada interna)
        particiones = StratifiedKFold(n_splits=self.n_particiones, shuffle=True, random_state=self.random_state)
        caracteristicas = None
        for idx_ent, idx_val in particiones.split(textos, etiquetas):
            comp = self._entrenar_componentes(textos.iloc[idx_ent], etiquetas.iloc[idx_ent])
            fila_val = self._caracteristicas(textos.iloc[idx_val], comp)
            if caracteristicas is None:
                caracteristicas = np.zeros((len(textos), fila_val.shape[1]))
            caracteristicas[idx_val] = fila_val

        self.etapa2_ = Pipeline([('escala', StandardScaler()),
                                 ('modelo', LogisticRegression(C=self.C_etapa2, max_iter=3000))])
        self.etapa2_.fit(caracteristicas, etiquetas)
        self.componentes_ = self._entrenar_componentes(textos, etiquetas)
        return self

    def predict_proba(self, X):
        textos = pd.Series(np.asarray(X['text']))
        return self.etapa2_.predict_proba(self._caracteristicas(textos, self.componentes_))

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

In [14]:
grilla_de = {'C_etapa2': [0.1, 1]}
busqueda_de = GridSearchCV(ModeloDosEtapasNBLR(), grilla_de, cv=kfold, scoring='accuracy', n_jobs=-1)
busqueda_de.fit(X_train, y_train)

print('Mejores hiperparámetros:', busqueda_de.best_params_)
print(f'Exactitud en validación cruzada: {busqueda_de.best_score_:.4f}   (modelo actual: {cv_actual.mean():.4f} | NB-LR: {busqueda_nblr.best_score_:.4f})')
pd.DataFrame(busqueda_de.cv_results_)[['param_C_etapa2', 'mean_test_score', 'std_test_score', 'rank_test_score']].round(4)

Mejores hiperparámetros: {'C_etapa2': 0.1}
Exactitud en validación cruzada: 0.9003   (modelo actual: 0.8948 | NB-LR: 0.8970)


,param_C_etapa2,mean_test_score,std_test_score,rank_test_score
0,0.1,0.9003,0.0075,1
1,1.0,0.8998,0.0080,2


## 7. Evaluación en el conjunto de prueba

In [15]:
mejor_de = busqueda_de.best_estimator_
pred_nblr = pd.Series(mejor_nblr.predict(X_test), index=y_test.index)
pred_de = pd.Series(mejor_de.predict(X_test), index=y_test.index)

display(pd.DataFrame({'Modelo actual': metricas(y_test, pred_actual),
                      'Modelo actual corregido': metricas(y_test, pred_corregido),
                      'NB-LR': metricas(y_test, pred_nblr),
                      'Dos etapas + NB-LR': metricas(y_test, pred_de)}).T.round(4))
print(classification_report(y_test, pred_de, digits=3))

fig, ax = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for eje, (titulo, pred) in zip(ax, [('Modelo actual', pred_actual), ('Modelo actual corregido', pred_corregido), ('Dos etapas + NB-LR', pred_de)]):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=ORDEN_CLASES, cmap='Blues', ax=eje, colorbar=False)
    eje.set_title(titulo)
plt.show()

,Exactitud,Precisión (macro),Recall (macro),F1 (macro)
Modelo actual,0.8946,0.8987,0.8998,0.8993
Modelo actual corregido,0.8954,0.8995,0.9005,0.9000
NB-LR,0.8983,0.9026,0.9035,0.9029
Dos etapas + NB-LR,0.8988,0.9037,0.9038,0.9037


              precision    recall  f1-score   support

    negativo      0.855     0.861     0.858       843
     neutral      0.997     0.999     0.998       715
    positivo      0.859     0.852     0.855       842

    accuracy                          0.899      2400
   macro avg      0.904     0.904     0.904      2400
weighted avg      0.899     0.899     0.899      2400



En el conjunto de prueba los tres modelos nuevos superan al modelo actual (0,8946): el modelo corregido llega a 0,8954, NB-LR a 0,8983 y el modelo en dos etapas con NB-LR a 0,8988 (F1-macro de 0,9037). Las diferencias son de pocas reseñas sobre 2.400, así que para compararlos de verdad hace falta la comparación con 15 particiones de la siguiente sección.

## 8. Comparación justa con 15 particiones

Un solo conjunto de prueba de 2.400 reseñas tiene un error de cerca de 0,6 puntos, así que para comparar modelos que difieren en décimas de punto se usan **las mismas 15 particiones que en el notebook de la Mejora 4** (3 repeticiones de validación cruzada de 5 particiones sobre las 12.000 reseñas, semillas 1, 2 y 3). Como todos los modelos se evalúan sobre exactamente las mismas particiones, se pueden comparar partición por partición y las cifras son directamente comparables con las de la Mejora 4.

In [16]:
# Las mismas 15 particiones del notebook de la Mejora 4: 3 repeticiones de 5 particiones sobre las 12.000 reseñas
particiones = []
for semilla in [1, 2, 3]:
    particiones += list(StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla).split(X, y))

modelos_justa = {
    'Modelo actual (regresión logística)': clone(modelo_actual),
    'Modelo actual corregido': clone(modelo_corregido),
    'NB-LR': clone(mejor_nblr),
    'Dos etapas + NB-LR': clone(mejor_de),
}
exactitudes = {}
for nombre, modelo in modelos_justa.items():
    exactitudes[nombre] = cross_val_score(modelo, X, y, cv=particiones, scoring='accuracy', n_jobs=-1)

ref = exactitudes['Modelo actual (regresión logística)']
filas = []
for nombre, valores in exactitudes.items():
    diferencia = valores - ref
    filas.append({'Modelo': nombre, 'Exactitud media': valores.mean(), 'Desviación': valores.std(ddof=1),
                  'Diferencia vs actual': diferencia.mean(),
                  'Error estándar de la diferencia': diferencia.std(ddof=1) / np.sqrt(len(diferencia)),
                  'Mejora en (de 15)': int((diferencia > 0).sum())})
tabla_justa = pd.DataFrame(filas).set_index('Modelo').sort_values('Exactitud media', ascending=False)
display(tabla_justa.round(4))

for a, b in [('Dos etapas + NB-LR', 'NB-LR'), ('Dos etapas + NB-LR', 'Modelo actual corregido'), ('NB-LR', 'Modelo actual corregido')]:
    dif = exactitudes[a] - exactitudes[b]
    print(f'{a} vs {b}: diferencia media {dif.mean():+.4f} '
          f'(error estándar {dif.std(ddof=1) / np.sqrt(len(dif)):.4f}), mejora en {(dif > 0).sum()} de 15 particiones')

,Exactitud media,Desviación,Diferencia vs actual,Error estándar de la diferencia,Mejora en (de 15)
Modelo,,,,,
Dos etapas + NB-LR,0.9023,0.0053,0.0056,0.0010,14
NB-LR,0.9004,0.0060,0.0037,0.0013,13
Modelo actual corregido,0.8988,0.0061,0.0021,0.0003,15
Modelo actual (regresión logística),0.8966,0.0062,0.0000,0.0000,0


Dos etapas + NB-LR vs NB-LR: diferencia media +0.0019 (error estándar 0.0007), mejora en 11 de 15 particiones
Dos etapas + NB-LR vs Modelo actual corregido: diferencia media +0.0035 (error estándar 0.0011), mejora en 13 de 15 particiones
NB-LR vs Modelo actual corregido: diferencia media +0.0016 (error estándar 0.0013), mejora en 9 de 15 particiones


Con las 15 particiones:

- **Dos etapas + NB-LR** obtiene una exactitud media de **0,9023**: 0,56 puntos más que el modelo actual (error estándar de 0,10) y mejora en 14 de las 15 particiones.
- **NB-LR** obtiene 0,9004 (+0,37 puntos, mejora en 13 de 15).
- **El modelo actual corregido** obtiene 0,8988 (+0,21 puntos, mejora en las 15 particiones).

Entre los nuevos, dos etapas + NB-LR supera a NB-LR por 0,19 puntos (11 de 15 particiones) y al modelo corregido por 0,35 (13 de 15). En el notebook de la Mejora 4, con estas mismas particiones, dos etapas + SVM obtuvo 0,9009, el modelo en dos etapas de la Mejora 1 0,8997 y el modelo actual 0,8966. **Dos etapas + NB-LR es el mejor modelo medido hasta ahora**, aunque su ventaja sobre la Mejora 4 es de solo 0,14 puntos.

## 9. ¿Dónde se siguen equivocando?

In [17]:
def clave_oracion(oracion):
    return re.sub(r'[^\w\s]', '', quitar_tildes(oracion.lower())).strip()


finales = Counter(clave_oracion(separar_oraciones(t)[-1]) for t in X_train['text'])
cierres = {o for o, n in finales.items() if n >= 10}
termina_en_cierre = X_test['text'].map(lambda t: clave_oracion(separar_oraciones(t)[-1]) in cierres)
es_polar = y_test != 'neutral'

grupos = {'Terminan en un cierre repetido': es_polar & termina_en_cierre,
          'Positivas/negativas que terminan en otra oración': es_polar & ~termina_en_cierre,
          'Neutrales': ~es_polar}
filas = []
for nombre, mascara in grupos.items():
    fila = {'Grupo': nombre, 'Reseñas': int(mascara.sum())}
    for modelo, pred in [('actual', pred_actual), ('corregido', pred_corregido), ('NB-LR', pred_nblr), ('dos etapas + NB-LR', pred_de)]:
        fila[f'Exactitud {modelo}'] = (pred == y_test)[mascara].mean()
    fila['Errores dos etapas + NB-LR'] = int((pred_de != y_test)[mascara].sum())
    filas.append(fila)
pd.DataFrame(filas).set_index('Grupo').round(3)

,Reseñas,Exactitud actual,Exactitud corregido,Exactitud NB-LR,Exactitud dos etapas + NB-LR,Errores dos etapas + NB-LR
Grupo,,,,,,
Terminan en un cierre repetido,198,0.460,0.475,0.495,0.495,100
Positivas/negativas que terminan en otra oración,1487,0.902,0.902,0.903,0.905,142
Neutrales,715,0.999,0.997,1.000,0.999,1


La mejora se concentra en las reseñas positivas y negativas. En las que terminan en un cierre ambiguo la exactitud sube de 0,460 a 0,495, pero sigue al nivel del azar: esas 198 reseñas producen 100 de los 243 errores del mejor modelo en el conjunto de prueba (41%). En el resto de las positivas y negativas sube de 0,902 a 0,905, y las neutrales siguen casi perfectas.

## 10. Cómo leer el puntaje público de Kaggle

La competencia calcula el puntaje público sobre solo 900 reseñas (el 30% de `eval.csv`); el ranking final normalmente se calcula con las 2.100 restantes. Con 900 reseñas el puntaje público tiene mucho ruido, y además elegir el mejor de varios envíos por su puntaje público lo infla. La siguiente celda cuantifica ambos efectos.

In [18]:
n_publico, n_privado = 900, 2100
for exactitud in [0.897, 0.904]:
    error = np.sqrt(exactitud * (1 - exactitud) / n_publico)
    print(f'Exactitud real {exactitud}: el puntaje público puede caer entre {exactitud - 1.96 * error:.3f} y {exactitud + 1.96 * error:.3f} (95%)')
print(f'En el privado (2.100 reseñas) el margen es de +/- {1.96 * np.sqrt(0.9 * 0.1 / n_privado):.3f}')

# Si un grupo hace muchos envíos de modelos igual de buenos, ¿cuánto sube su mejor puntaje público solo por suerte?
generador = np.random.default_rng(0)
for envios in [5, 20]:
    mejores = generador.binomial(n_publico, 0.90, size=(20000, envios)).max(axis=1) / n_publico
    print(f'{envios:2d} envíos con exactitud real 0.900 -> mejor puntaje público promedio: {mejores.mean():.4f}')

Exactitud real 0.897: el puntaje público puede caer entre 0.877 y 0.917 (95%)
Exactitud real 0.904: el puntaje público puede caer entre 0.885 y 0.923 (95%)
En el privado (2.100 reseñas) el margen es de +/- 0.013
 5 envíos con exactitud real 0.900 -> mejor puntaje público promedio: 0.9115
20 envíos con exactitud real 0.900 -> mejor puntaje público promedio: 0.9183


Con 900 reseñas, un modelo con exactitud real de 0,904 puede obtener en el público cualquier puntaje entre 0,885 y 0,923 (95%). Además, si un grupo hace muchos envíos de modelos igual de buenos y se queda con el mejor puntaje público, ese puntaje sube solo por suerte: con 20 envíos de exactitud real 0,900, el mejor sale en promedio 0,918. En la práctica el efecto es menor, porque los envíos de un mismo grupo se parecen entre sí, pero existe.

Por eso los modelos se eligen con la comparación de 15 particiones y no con el puntaje público. Por ejemplo, el ajuste fino de la Mejora 3 obtuvo 0,90666 en el público (816 de 900), el mejor puntaje del grupo, pero en las 15 particiones no se distingue del modelo actual.

## 11. Modelos finales y predicciones para la competencia

Los tres modelos nuevos se vuelven a entrenar con las 12.000 reseñas etiquetadas y se guardan con sus envíos. Se comparan con el envío del modelo actual y con el del ajuste fino, que es el que tiene el mejor puntaje público del grupo.

In [19]:
envio_actual = pd.read_csv(os.path.join(RUTA_ENVIOS, 'submission_regresion_logistica_bloques.csv'))
envio_fino = pd.read_csv(os.path.join(RUTA_ENVIOS, 'submission_regresion_logistica_ajuste_fino.csv'))

finales_modelos = {NOMBRE_CORREGIDO: clone(modelo_corregido), NOMBRE_NBLR: clone(mejor_nblr), NOMBRE_MODELO: clone(mejor_de)}
envios = {}
for nombre, modelo in finales_modelos.items():
    modelo.fit(X, y)
    envio = pd.DataFrame({'id': evaluacion['id'], 'answer': modelo.predict(evaluacion[['text']])})
    envio.to_csv(os.path.join(RUTA_ENVIOS, f'submission_{nombre}.csv'), index=False)
    joblib.dump(modelo, os.path.join(RUTA_MODELOS, f'{nombre}.joblib'))
    envios[nombre] = envio
    print(f'{nombre}: formato correcto = {list(envio.columns) == list(muestra_envio.columns) and (envio["id"] == muestra_envio["id"]).all()} | '
          f'distintas vs modelo actual: {(envio["answer"] != envio_actual["answer"]).sum()} | '
          f'distintas vs ajuste fino: {(envio["answer"] != envio_fino["answer"]).sum()}')

display(pd.DataFrame({nombre: e['answer'].value_counts(normalize=True) for nombre, e in envios.items()}).reindex(ORDEN_CLASES).round(3))

regresion_logistica_corregida: formato correcto = True | distintas vs modelo actual: 20 | distintas vs ajuste fino: 40


nb_regresion_logistica: formato correcto = True | distintas vs modelo actual: 185 | distintas vs ajuste fino: 169


dos_etapas_nblr: formato correcto = True | distintas vs modelo actual: 145 | distintas vs ajuste fino: 135


,regresion_logistica_corregida,nb_regresion_logistica,dos_etapas_nblr
answer,,,
negativo,0.355,0.355,0.362
neutral,0.281,0.282,0.277
positivo,0.364,0.363,0.361


In [20]:
for nombre in finales_modelos:
    cargado = joblib.load(os.path.join(RUTA_MODELOS, f'{nombre}.joblib'))
    iguales = (cargado.predict(evaluacion[['text']]) == envios[nombre]['answer']).all()
    print(f'{nombre}: el modelo guardado reproduce las mismas predicciones -> {iguales}')

regresion_logistica_corregida: el modelo guardado reproduce las mismas predicciones -> True


nb_regresion_logistica: el modelo guardado reproduce las mismas predicciones -> True


dos_etapas_nblr: el modelo guardado reproduce las mismas predicciones -> True


El modelo corregido cambia solo 20 predicciones respecto al modelo actual; NB-LR cambia 185 y dos etapas + NB-LR 145. La distribución de clases predichas es parecida a la de entrenamiento en los tres casos. Los modelos guardados reproducen exactamente las predicciones.

## 12. Conclusiones

- **Había un error en la representación del notebook principal.** Cortar la cláusula final en "al final de cuentas" la dejaba vacía en cerca del 3,5% de las reseñas. Corregirlo mejora al modelo actual en 0,21 puntos de forma consistente (15 de 15 particiones). Es la mejora más simple de explicar.
- **NB-LR es una técnica clásica que sí ayuda.** Ponderar la bolsa de palabras con las razones de Naive Bayes antes de la regresión logística mejora 0,37 puntos sobre el modelo actual.
- **El mejor modelo medido es dos etapas + NB-LR**, con 0,9023 de exactitud en 15 particiones (+0,56 sobre el modelo actual). Su ventaja sobre la Mejora 4 (0,9009) es pequeña.
- **El techo está cerca.** Las reseñas con cierre ambiguo siguen al nivel del azar y producen cerca del 40% de los errores. Todas las ideas probadas terminan corrigiendo, en parte, los mismos errores.
- **Para Kaggle**, se espera que dos etapas + NB-LR rinda alrededor de medio punto más que el modelo actual en el ranking final, aunque el puntaje público, con 900 reseñas, puede no reflejarlo.

## 13. Uso de herramientas de IA generativa

**Declaración del uso.** Se utilizó Claude Code (modelo Claude Sonnet 5.5, de Anthropic) como apoyo para: proponer y generar un primer esqueleto del código de este notebook, depurar errores y redactar las explicaciones.

**Prompts utilizados.**

1. *"tenemos que si o si encontrar una manera de construir un modelo mejor, en el kaggle publico la medida más alta que tenemos es de 0,9066 para submission_regresion_logistica_ajuste_fino.csv y ya hay compañeros que van en 0,9155"*. A partir de ese prompt la herramienta exploró varias alternativas (detección de opiniones por aspecto, NB-LR, segundas etapas con boosting y Random Forest, n-gramas de caracteres) y propuso las dos de este notebook, que fueron las únicas que mejoraron de forma consistente.

**Análisis crítico del resultado.**

[Completar por el equipo: ¿qué partes del código y de las explicaciones generadas fueron correctas y útiles? ¿qué errores, imprecisiones o limitaciones se identificaron? ¿qué decisiones técnicas se cambiaron respecto a lo que propuso la herramienta y por qué? ¿qué conceptos del curso permitieron evaluar o corregir la respuesta?]

**Aportes propios del equipo.**

[Completar por el equipo: qué se desarrolló, modificó o decidió, qué ajustes se hicieron al código y a las explicaciones, y qué se aprendió del proceso.]

## 14. Cómo replicar los resultados

1. Crear un entorno con Python 3.9 e instalar las librerías con `pip install -r requirements.txt` (están en la raíz del proyecto).
2. Mantener la estructura de carpetas del proyecto: este notebook está en `notebooks/`, los datos en `data/`. El notebook se ejecuta desde la carpeta `notebooks/`.
3. Ejecutar todas las celdas en orden (`Restart & Run All`). Tarda aproximadamente 22 minutos.
4. Al terminar se generan dos modelos y dos envíos: `models/nb_regresion_logistica.joblib` y `models/dos_etapas_nblr.joblib`, con sus archivos `submissions/submission_nb_regresion_logistica.csv` y `submissions/submission_dos_etapas_nblr.csv`.

Todas las fuentes de aleatoriedad usan semillas fijas, por lo que las cifras se reproducen exactamente con las versiones de `requirements.txt`. Para cargar el modelo guardado fuera del notebook hay que ejecutar antes las celdas que definen las funciones y las clases `NBRegresionLogistica` y `ModeloDosEtapasNBLR` usadas por el pipeline.